In [1]:
%pylab inline
from inu.ds import *
from inu.ds.transform import apply_transforms, align_transformed
from inu.visual.insight import *

import pandas as pd

# FORMATTING SETTINFS
from qbstyles import mpl_style
mpl_style(dark=True)

pd.set_option('display.max_colwidth', 20)
np.set_string_function(array_str)   # show only arrays info 

Populating the interactive namespace from numpy and matplotlib


# Datasets and `PathSchemes`
Structures of datasets and their formal definitions using `PathScheme` are described elsewhere  
(see, `inu.ds` docs and `tests/examples.ipynb` notebook).

Here we just remind that `inu.ds` package considers:
  - data set as a collection of _data items_, 
  - each item as collection of labels `{category: value}`
  
### Specific Labeling Conventions:
Those are default for the framework (can be changed):
 - `path` - location of the items data 
 - `data` - when loaded as an object into memory
 - `transforms` - used by scheme to define transforms required to "normalize" the data
 - `transformed` - desribes how data _has been_ transformed from the "normal" state
 
Those are more domain specific:

 - `alg`  - category is a _method_ used to produce the data.   
 - `kind` - provides additional specifics about its nature

## From `PathScheme`s to `DataCollection`
Processing of experiments may require coordinated acces to _multiple_ related datasets.  
In this example we have two datasets:
   1. The original stereo dataset (Middlebury) with _left, right_ images and _disparity_ Ground Truth
   2. Disparities calculated from the Middlebury stereo pairs by different algorithms 

Analysis of those results require coordinated access to both data sets, when, for example,  
for every _scene_ the ground truth (GT) and disparities of evaluated algorithms must be compaired.

`DataCollection` class combines those datasets, to allow flexible queries, iterations,  
arbitrary data transformations, and other data logistics capabilities.

In this case schemes are stored in the respective folder of every dataset,   
so we can initialize the schemes by those folders:

In [2]:
datasets = ['/home/ilyap/work/data/depth/mid', 
           '/home/ilyap/work/data/evals/comp_nets_202803/mid']
schemes = [*map(PathScheme.from_yaml, datasets)]

The structure of the datsets can be examined, to see which `{categories}` are defined,  
and if there are some common ones:

In [3]:
print('\n'.join(map(lambda s: f"{s} defines: {s.categories}", schemes)))
s1, s2 = schemes
print("Common categories:", f"{s1.categories.intersection(s2.categories)}")

PathScheme<{scene}/{kind}{side}{alg}.{ext}> defines: {'kind', 'side', 'scene', 'rgn_codes', 'alg', 'ext'}
PathScheme<{scene}/{alg}.{ext}> defines: {'kind', 'transforms', 'transformed', 'ext', 'side', 'scene', 'rgn_codes', 'alg'}
Common categories: {'kind', 'scene', 'ext', 'side', 'rgn_codes', 'alg'}


## Creating DataCollection

### Data Structure
`DataCollection` gathers elements from all the datasets as a single table (`pandas.DataFrame`) with  
 - categories as __columns__, and
 - data items as __rows__

Logically the categories can be separated into the _index_ and _data_ parts.  

 - __index__ uses labels more suitable for querying, while
 - __data__ is usually meant for processing   

though the structure is flexible, can be changed dynamically, and even some categories could be assigned to both.  
Technically, the multi-category index is implemented as `pandas.MultiIndex` for the rows.

By default all categories are set as _index_ except the `path`, `transfomed`, `transforms` and `data`.


### Cleanups
Meaningful merge of datasets requires all the common categories to share sematical meaning.  
Those, and also categories not needed for specific work should be **dropped** .   
Basic cleanup may also include filtering out certain rows (unneeded data elements).  
In this case, that would be certain masks `alg==nocc` provided by the Middlebury datasets.   

In [4]:
dc = DataCollection(schemes, dataset='mid evals',                # source
                    query="alg!='nocc'", drop=['dataset', 'ext'])   # content
dc

Collection 'mid evals' 142 items
[scene,kind,side,alg] -> [path,rgn_codes,transforms,transformed]

The table is accessible as a public attribute `DataCollection.db` .   
The few rows the table below show its structure: 4 levels of index and 3 data columns:  

(Tables visualization can be improved by patching their representation `df=p_*df` or printing `p_|df`)

In [5]:
Formatters.html_as_text(False)
dc.db[::9]

,,,,path,rgn_codes,transforms,transformed
scene,kind,side,alg,,,,
ArtL,image,right,,…th/mid/ArtL/im1...,NaN,NaN,NaN
Teddy,mask,left,occl,…Teddy/mask0nocc...,"{'invl': 0, 'occ...",NaN,NaN
Pipes,image,right,,…h/mid/Pipes/im1...,NaN,NaN,NaN
Piano,mask,left,occl,…Piano/mask0nocc...,"{'invl': 0, 'occ...",NaN,NaN
Playtable,image,right,,…d/Playtable/im1...,NaN,NaN,NaN
Jadeplant,mask,left,occl,…plant/mask0nocc...,"{'invl': 0, 'occ...",NaN,NaN
Shelves,image,right,,…mid/Shelves/im1...,NaN,NaN,NaN
Vintage,mask,left,occl,…ntage/mask0nocc...,"{'invl': 0, 'occ...",NaN,NaN
ArtL,disp,left,DPrune,…DeepPruner_disp...,NaN,"squeeze, norm(Tr...",NaN


# Working with Data
## Normalization and Transformations 
There are two major and rather standard challenges when working with large datasets:
 1. The volume of its entire data may too large to load into the memory
 2. The raw data is not ready for homogeneous processing pipeline
 
Second point referes to misalignment in characteristics and representation of the data,  
preventing uniform processing of different data items of same kind, for multiple possible reasons:
 - images can be cropped differently, or packed differently in multi-dimentional arrays
 - data can be measured in different units, or bits representations
Even worse, those alterations and variations are often implicit, supposedely _known_ or written _somewehere_!

This framewrork suggests a structural approach this problem:
 - Make implicit explicit - dataset must be self-contained and describe all the "alterations"
 - Consider data normalization (or alignment) as a standard stage of the processing pipeline

Then those (and other) challenges can be approached using _transformations_.  
A transformation is a general operation on _data columns_:   

    `{x, y ... z} -> T -> {u, v ... w}`

That is new data lebels are created from existing (or overwrite them).  

## Complete Dataset Scheme
Scheme of the `comp_nets_202803` below describes several aspects of the dataset:
 - `pattern` defines file-system structure and associated labeles extraction rules
 - `alias`   renames values of _some_ of the _alg_ labels to follow convention
 - explicitely adds lablels not deined otherwise:
     - this dataset _happens_ to include only `disp`arity `left` images!
     - this critical information for processing usually appears as dataset-dependent code inserts
 - explicit _conditional_ instructions defining data-transfomations:
     - disparities produced by _some_ algorithms are scaled 
     - outputs of _some other_ algorithms are cropped from the original in certain way     
     - _conditional labeling supports two syntaxes, allowing to group by condition or category_

This particular scheme introduces two transforms-related categories: `transforms` and `transfomed` .  
Those are default names expected by the `DataCollection` class but not mandatory.
This default convention expects 
  - `transforms` to define direct instructions normalizing the data (reversing the alteration)
  - `transormed` is used when inverse transform is not available, but it _could_ be applied to align the others

In [26]:
# %load /home/ilyap/work/data/depth/mid/info.yml
description: Subset of the Middlebury Stereo Dataset 
dataset: Middlebury 2014
domain: stereo

scheme:
    pattern: '{scene}/(?P<kind>[^\d\\/]+)(?P<side>\d)(?P<alg>\w*)\.(?P<ext>pfm|png)'
    alias:
        side: {"0": L, "1": R}
        kind: {im: image, mask: regns}
        alg:  {nocc: occl}  # occlusion regions

    if alg in {'occl'}:     # `occl` is a multi-level regions map
        rgn_codes: {invl: 0, occl: 128, nocc: 255}  # with codes

    transforms if alg == 'GT':
        recode:
            from_to: [.inf, .nan]     # recode .inf -> .nan



In [25]:
# %load /home/ilyap/work/data/evals/comp_nets_202803/mid/scheme.yml
pattern: '{scene}/(?P<alg>[-\w]+)\.(?P<ext>tif|pfm|png)'
alias:
    alg:
        AcfNet_out_disp:     Acf
        DeepPruner_disp:     DPrune
        disp0HSM:            HSM
        GANet-deep_out_disp: GADeep
        out_disp:            Stereo          

# assign labels below unconditionally to all the items 
kind: disp
side: L

transforms:                             # some images could have been in tensor dimensions
    squeeze: {}                         # apply the squeeze on all the images

# assign labels below if condition holds:                                
if alg in {'GADeep', 'DPrune'}:         # query define when the labels below to be assigned 
    transforms:                        # Label's category, followed by it's value:     
        norm:                           # data undergone `norm` transformation
            inverse: True
            scale: 256                  # file_data = (data - offset) * scale             
                                      
# labeling condition may be also defined as a part of the label:
transformed if alg in {'GADeep'}:       # transform has been applied to GADeep
    center_crop:                        # name of the transform
        width: 1200                     # named arguments of the transform
        height: 816

if alg == 'HSM':                        # this algorithm uses
    recode: [.inf, .nan]                # infinity to encode invalid pixels

transforms if alg == 'HSM':
    recode:
        from_to: [.inf, .nan]


## Multi-Level API 
Pandas provides very powerfull toolset for re-arranging, subsetting and processing tables.  
That was a motivation to implement the `DataCollection.db` table as `pandas.DataFrame`, and to expose it to the user.  
The `DataCollection` itself provides quite thin API for the most common use-cases, so that  
user is expected to master `pandas` and use its power in the multitude of possible scenarious.

### Iterations
The most important and common use case is to iterate over certain groups of data items and apply certain processing.  
`DataCollection.iter` is a very powerfull and versatyle function, allowing to fully define the structure of such groups.  

Example below creates an iterator over subsets _grouped by scenes_ indexed by specified categories `['kind', 'alg', 'side']` .   
By default it also applies standard set of __transformations__ and __drops__ itermediate data stages:

    [path] -> read -> [data] -> transforms -> [data] -> tranformed -> [data]

In [7]:
scene, grp = g = next(dc.iter('scene', ['kind', 'alg', 'side']))
print(f"       Scene [{scene}]", grp, sep='\n')

       Scene [Adirondack]
                              rgn_codes             data
kind  alg    side                                       
image        right                  NaN  [816⨉1200⨉3] u1
disp  GT     left         {'invl': inf}    [816⨉1200] f4
             right        {'invl': inf}    [816⨉1200] f4
mask  occl   left   {'invl': 0, 'occ...    [816⨉1200] u1
             right  {'invl': 0, 'occ...    [816⨉1200] u1
image        left                   NaN  [816⨉1200⨉3] u1
disp  DPrune left                   NaN    [816⨉1200] f8
      Stereo left                   NaN    [816⨉1200] f4
      HSM    left         {'invl': inf}    [816⨉1200] f4
      PDS    left                   NaN    [816⨉1200] f4
      Acf    left                   NaN    [816⨉1200] f4
      GADeep left                   NaN    [816⨉1200] f8


If some item of the group are not required we may filter them out:

In [8]:
filter_left = lambda g: g.xs('left', level='side', drop_level=False)
itr = dc.iter('scene',['kind', 'alg', 'side']) / filter_left
next(itr)

gid: Adirondack, grp:
                             rgn_codes             data
kind  alg    side                                      
disp  GT     left        {'invl': inf}    [816⨉1200] f4
mask  occl   left  {'invl': 0, 'occ...    [816⨉1200] u1
image        left                  NaN  [816⨉1200⨉3] u1
disp  DPrune left                  NaN    [816⨉1200] f8
      Stereo left                  NaN    [816⨉1200] f4
      HSM    left        {'invl': inf}    [816⨉1200] f4
      PDS    left                  NaN    [816⨉1200] f4
      Acf    left                  NaN    [816⨉1200] f4
      GADeep left                  NaN    [816⨉1200] f8

To take full control over the transformations we may diable them in `iter` and apply __explicitly__ using chaining operator `/`:

In [13]:
itr = (dc.iter('scene',['kind', 'alg', 'side'], trans=False)
       / filter_left
       / (lambda g: apply_transforms(g, out='temp', keep='path'))
       / (lambda g: align_transformed(g, inp='temp', out='data', keep='temp')))
scene, grp = next(itr)
grp

rgn_codes                 path             temp  \
kind  alg    side                                                              
disp  GT     left        {'invl': inf}  …rondack/disp0GT...    [992⨉1436] f4   
mask  occl   left  {'invl': 0, 'occ...  …ndack/mask0nocc...    [992⨉1436] u1   
image        left                  NaN  …/Adirondack/im0...  [992⨉1436⨉3] u1   
disp  DPrune left                  NaN  …DeepPruner_disp...    [992⨉1436] f8   
      Stereo left                  NaN  …ondack/out_disp...    [992⨉1436] f4   
      HSM    left        {'invl': inf}  …ondack/disp0HSM...    [992⨉1436] f4   
      PDS    left                  NaN  …/Adirondack/PDS...    [992⨉1436] f4   
      Acf    left                  NaN  …AcfNet_out_disp...    [992⨉1436] f4   
      GADeep left                  NaN  …t-deep_out_disp...    [816⨉1200] f8   

                              data  
kind  alg    side                   
disp  GT     left    [816⨉1200] f4  
mask  occl   left    [816⨉1200] u1  
image        left  [816⨉1200⨉3] u1  
disp  DPrune left    [816⨉1200] f8  
      Stereo left    [816⨉1200] f4  
      HSM    left    [816⨉1200] f4  
      PDS    left    [816⨉1200] f4  
      Acf    left    [816⨉1200] f4  
      GADeep left    [816⨉1200] f8

## Accessing Data Elements
`DataFrame` provides flexible access to its columns, rows and index levels as to attributes and as dict keys:

In [20]:
grp.iloc[[0]]  # show slice of rows with indices in [0],  iloc[0] - would be the row itself

,,,rgn_codes,path,temp,data
kind,alg,side,,,,
disp,GT,left,{'invl': inf},…rondack/disp0GT...,[992⨉1436] f4,[816⨉1200] f4


In [11]:
(grp.data[1]
 is grp.data[('disp', 'GT', 'left')]
 is grp.data.disp.GT.left 
 is grp['data'].disp['GT'].left)

True

That allows expressive and concise syntax and integration with other functions: 

In [ ]:
for i, (scene, grp) in enumerate(itr):    
    if i >= 2: break;         
    imgrid(grp.data.reindex(['image', 'disp'], level='kind'), 
           clim=[0, 1], mosaic='titles')  # reindex to place `image` before `disp`

# Building Workflows
## Assumptions

There are some general assumptions which once satisfied simplifies greately code and workflows:
 - Invalid data elements `invl` are represented by same value (`np.nan`) 
    - `np.nan` is a preferable choice for its wide support by `numpy` and `pandas`
    - necessary exceptions may include `integer` data types, where special traetment is required
      (pandas started to support `np.nan` even in integer arrays!)
    - invalids are by default excluded from standard calculations (metrics, etc)    
 - Data arrays are spatially aligned - same mapping between their indices and `x,y`

## Stable Labeling
The datasets framework `inu.ds` is quite generic and makes very few assumptions.  
However smooth work with data requires simplicity and stability not only of its _explicit API_ ,  
but also of the _implicit_ one - the data structure itself, defined mainly the _data labels_.  

Therefore its important to follow two general guidelines:
1. Turn all _implicit_ knowledge about the dataset into _explicit_ through `schemes`.
2. Make the _labeling_ and codes of categories _stable_.

Labeling conventions are expected to be specific to a particular application domain.  
A special class `inu.ds.LabelRules` comes to facilitate stabilization of the labeling for every domain.

In [39]:
print("Registered domains:", LabelRules.domains())

Registered domains: ['stereo']


In [42]:
rules = LabelRules(domain='stereo')
rules

Labeling Rules for stereo domain
Labels categories: ['units', 'msr', 'alg', 'kind', 'side', 'rgn']

In [46]:
'msr' in rules, rules.in_cat('side', 'left'), rules.in_cat('side', 'top')

(True, True, False)

In [44]:
rules.labels['side'], rules.labels['kind']

({'left', 'right'}, {'cnfd', 'depth', 'disp', 'image', 'mask', 'model'})

In [45]:
rules.labels['alg'], rules.in_cat('alg', 'Anything is allowed')

(set(), True)

## Typical Workflow Components
Various data processing scenarios, especially in a specific domain, share stages and concepts,  
which, once formalized, streamline thinking and coding.

Constructs presented below are typical for the stereo domain, but also way beyond.

### Regions Maps
Elements in data (like pixels in image) may be associated with different semantic classes.  
This information is defined using _region maps_ .    
In general form a _region map_ is associated with every _region kind_ (semantic class):  `{region_kind: region_map}`  

If classes are not itersected a _single region map_ may be used with regions differentiated by _region codes_ .      
That also asumes that association to the classes is _binary_, that is a pixel is either belongs to a class or not.  

Sometimes semantic 'painting' is multi-level, and the region maps are not represented by binary masks.  
Multi-level classes could be collapsed into binary for simplification.

   > For region-based metrics read on [...Evaluation of Stereo Algorithms](https://vision.middlebury.edu/stereo/taxonomy-IJCV.pdf)

### Some common semantic codes:

| Code   | bin | Semantics   | Comments  
|--------|:---:|-------------|---------------
| `invl` |  Y  | no GT       | Semantic map version of invalids
| `occl` |  Y  | occlusion   | from the GT dataset or calculated
| `edge` |  N  | object edge | near the discontinuity in the depth of objects
| `tilt` |  N  | tilted      | Areas with normals tilted from the cameras axis 
| `txtr` |  N  | texture     | Spacial variability in the image (depends on threshold)      
| `cnfd` |  N  | confidence  | Some algorithms produce measure of their confidence
| `devt` |  N  | deviation   | Based on observed deviation from the GT

### Composed Regions
Independent region classes may combined into new categories: `bad_edge` = `err` & `dscn` .  
In general that can be described as operations on categorical sets: 
$$C_{bad\ edge} = C_{err} \cap C_{dscn}$$

## Uniform Metrics
In general terms a _metric_ as an arbitrary function corresponding a collection of elements to a single number. 
Most of the metrics are statistical function $F_S$ processing their input set $V$ _uniformly_,
that is __not sensitive to the order or spacial location__ of those values $M = F_S(V)$

This set $V$ of values generally is a product of chain of operatioins, for _example_  
starting from the input data, like 
   - images $I_{L,R}$, 
   - disparities $D_{L,R}$ 
   - ground truth disparity $G_{L,R}$        

then passing through some categorizations:
   $$\begin{aligned}
   C_{err} &= |G-D| > T_{err} \\
   C_{dscn} &= \mathrm{canny}(I)
   \end{aligned}$$
then through categories based filtering:
    $$ V = D \in (C_{err} \cap C_{dscn}) $$
to the final statiscical processing $F_S(V)$.

That allows to define a generic `uniform_metrics` function 
 - applying arbitrary _measurements_ 
 - _differentially_ by specific _regions_ in the data 
 
`M = uniform_metrics(input_set, measures, regions)`